In [0]:
# Import necessary libraries
from pyspark.sql import functions as F

In [0]:
# Define date range
start_date = spark.sql('SELECT MIN(date) FROM sportsdirect_sales.sportsdirect_gold.fact_order')
end_date = spark.sql('SELECT MAX(date) FROM sportsdirect_sales.sportsdirect_gold.fact_order')

In [0]:
# Check dates
display(start_date)
display(end_date)

MIN(date)
2024-01-01


MAX(date)
2025-12-01


In [0]:
# Create unique dates dataframe
dates = spark.sql('SELECT DISTINCT date FROM sportsdirect_sales.sportsdirect_gold.fact_order')

# Display dates dataframe
display(dates)

date
2024-12-01
2025-08-01
2025-04-01
2025-11-01
2024-02-01
2024-09-01
2024-04-01
2024-07-01
2024-01-01
2024-06-01


In [0]:
# Add analytics columns to the dates dataframe
dates = dates.withColumn('date_key', F.date_format('date', 'yyyyMM').cast('int')) \
        .withColumn('year', F.year('date')) \
        .withColumn('month', F.date_format('date', 'MMMM')) \
        .withColumn('month_short_name', F.date_format('date', 'MMM')) \
        .withColumn('quarter', F.concat(F.lit('Q'), F.quarter('date'))) \
        .withColumn('year_quarter', F.concat(F.col('year'), F.lit('-'), F.col('quarter')))

# Display dates dataframe
display(dates)

date,date_key,year,month,month_short_name,quarter,year_quarter
2024-12-01,202412,2024,December,Dec,Q4,2024-Q4
2025-08-01,202508,2025,August,Aug,Q3,2025-Q3
2025-04-01,202504,2025,April,Apr,Q2,2025-Q2
2025-11-01,202511,2025,November,Nov,Q4,2025-Q4
2024-02-01,202402,2024,February,Feb,Q1,2024-Q1
2024-09-01,202409,2024,September,Sep,Q3,2024-Q3
2024-04-01,202404,2024,April,Apr,Q2,2024-Q2
2024-07-01,202407,2024,July,Jul,Q3,2024-Q3
2024-01-01,202401,2024,January,Jan,Q1,2024-Q1
2024-06-01,202406,2024,June,Jun,Q2,2024-Q2


In [0]:
# Write the dates dataframe to the table
dates.write \
    .mode('overwrite') \
    .format('delta') \
    .saveAsTable('sportsdirect_sales.sportsdirect_gold.dim_date')